<a href="https://colab.research.google.com/github/miray7yuce/quadcopter-rl-copilot/blob/main/notebooks/quadcopter_rl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
!pip install -q stable-baselines3 gymnasium
!pip uninstall -y -q jsbsim
!pip install -q jsbsim==1.2.4

import jsbsim
print(jsbsim.__version__)

1.2.4


In [22]:
from google.colab import userdata
import os

USER  = "miray7yuce"
REPO  = "quadcopter-rl-copilot"
TOKEN = userdata.get('GH_TOKEN')

!git config --global user.email "miray7yuce@gmail.com"
!git config --global user.name "miray7yuce"

os.environ['REMOTE'] = f"https://{TOKEN}@github.com/{USER}/{REPO}.git"
!rm -rf /content/repo
!git clone -q $REMOTE /content/repo
!ls -a /content/repo

.   configs  .gitignore  README.md	   runs
..  .git     notebooks	 requirements.txt  src


In [21]:
!curl -fsSl https://deb.nodesource.com/setup_20.x | sudo -E bash -
!sudo apt-get install -y nodejs
!sudo npm install -g @anthropic-ai/claude-code

2026-08-31 10:05:02 - Installing pre-requisites
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://deb.nodesource.com/node_20.x nodistro InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
30 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' do

In [34]:
from google.colab import userdata
import os
os.environ["ANTHROPIC_API_KEY"]=userdata.get('tusaş')

In [23]:
!pip freeze | grep -iE "^(jsbsim|stable-baselines3|gymnasium|torch|numpy)=" > /content/repo/requirements.txt
!cat /content/repo/requirements.txt

gymnasium==1.3.0
jsbsim==1.2.4
numpy==2.1.3


In [24]:
import os, sys

BASE = "/content/repo"

for d in ["src/drone_rl/envs", "src/drone_rl/utils", "configs"]:
    os.makedirs(f"{BASE}/{d}", exist_ok=True)

for p in ["src/drone_rl", "src/drone_rl/envs", "src/drone_rl/utils"]:
    open(f"{BASE}/{p}/__init__.py", "a").close()

with open(f"{BASE}/.gitignore", "w") as f:
    f.write("__pycache__/\n*.zip\n*.pkl\nlogs/\nruns/\n.ipynb_checkpoints/\n")

sys.path.insert(0, f"{BASE}/src")

!find /content/repo -not -path '*/.git/*' -type f | sort

os.environ['PYTHONPATH'] = f"{BASE}/src"

/content/repo/configs/ppo_hover.yaml
/content/repo/.gitignore
/content/repo/notebooks/quadcopter_rl.ipynb
/content/repo/README.md
/content/repo/requirements.txt
/content/repo/runs/hover_v1/model.zip
/content/repo/runs/hover_v1/vecnormalize.pkl
/content/repo/src/drone_rl/envs/f450_env.py
/content/repo/src/drone_rl/envs/__init__.py
/content/repo/src/drone_rl/evaluate.py
/content/repo/src/drone_rl/__init__.py
/content/repo/src/drone_rl/train.py
/content/repo/src/drone_rl/utils/__init__.py
/content/repo/src/drone_rl/utils/units.py


In [25]:
%%writefile /content/repo/src/drone_rl/utils/units.py
"""Birim donusumleri. JSBSim emperyal birim kullanir."""

FT2M = 0.3048
M2FT = 1.0 / FT2M

def ft_to_m(x):
    return x * FT2M

def m_to_ft(x):
    return x * M2FT

Overwriting /content/repo/src/drone_rl/utils/units.py


In [12]:
%%writefile /content/repo/src/drone_rl/envs/f450_env.py
"""F450 quadcopter icin hover gorevi ortami."""

import numpy as np
import gymnasium as gym
from gymnasium import spaces
import jsbsim

HOVER_THROTTLE = 0.420
THROTTLE_RANGE = 0.25


class F450HoverEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, target_altitude_ft=30.0, episode_seconds=20.0,
                 physics_hz=240, control_hz=20):
        super().__init__()

        self.action_space = spaces.Box(-1.0, 1.0, shape=(4,), dtype=np.float32)
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(13,), dtype=np.float32)

        self.target_altitude = target_altitude_ft
        self.physics_dt = 1.0 / physics_hz
        self.substeps = int(physics_hz / control_hz)
        self.max_steps = int(episode_seconds * control_hz)

        self.fdm = jsbsim.FGFDMExec(None)
        self.fdm.set_debug_level(0)
        if not self.fdm.load_model("F450"):
            raise RuntimeError("F450 modeli yuklenemedi")
        self.fdm.set_dt(self.physics_dt)

        self.step_count = 0
        self.prev_action = np.zeros(4, dtype=np.float32)

    def _apply_initial_conditions(self):
        h0 = self.target_altitude + self.np_random.uniform(-3.0, 3.0)
        self.fdm["ic/h-agl-ft"] = h0
        self.fdm["ic/u-fps"] = self.np_random.uniform(-1.0, 1.0)
        self.fdm["ic/v-fps"] = self.np_random.uniform(-1.0, 1.0)
        self.fdm["ic/w-fps"] = self.np_random.uniform(-1.0, 1.0)
        self.fdm["ic/phi-rad"] = self.np_random.uniform(-0.05, 0.05)
        self.fdm["ic/theta-rad"] = self.np_random.uniform(-0.05, 0.05)
        self.fdm["ic/psi-true-rad"] = 0.0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self._apply_initial_conditions()
        self.fdm.run_ic()

        for i in range(4):
            self.fdm[f"propulsion/engine[{i}]/set-running"] = 1
        self.fdm["fcs/ScasEngage"] = 0

        for i in range(4):
            self.fdm[f"fcs/throttle-cmd-norm[{i}]"] = HOVER_THROTTLE

        self.step_count = 0
        self.prev_action = np.zeros(4, dtype=np.float32)

        return self._get_obs(), {}

    def _get_obs(self):
        f = self.fdm
        alt_err = (f["position/h-agl-ft"] - self.target_altitude) / 10.0
        hdot = f["velocities/h-dot-fps"] / 10.0
        u = f["velocities/u-fps"] / 10.0
        v = f["velocities/v-fps"] / 10.0
        roll = f["attitude/phi-rad"]
        pitch = f["attitude/theta-rad"]
        p = f["velocities/p-rad_sec"] / 5.0
        q = f["velocities/q-rad_sec"] / 5.0
        r = f["velocities/r-rad_sec"] / 5.0

        return np.array(
            [alt_err, hdot, u, v, roll, pitch, p, q, r, *self.prev_action],
            dtype=np.float32,
        )

    def step(self, action):
        #action = np.clip(np.asarray(action, dtype=np.float32), -1.0, 1.0)
        throttles = np.clip(HOVER_THROTTLE + action * THROTTLE_RANGE, 0.0, 1.0)

        for _ in range(self.substeps):
            for i in range(4):
                self.fdm[f"fcs/throttle-cmd-norm[{i}]"] = float(throttles[i])
            self.fdm.run()

        self.step_count += 1
        obs = self._get_obs()

        alt_err_ft = abs(self.fdm["position/h-agl-ft"] - self.target_altitude)
        tilt = abs(self.fdm["attitude/phi-rad"]) + abs(self.fdm["attitude/theta-rad"])
        spin = abs(self.fdm["velocities/p-rad_sec"]) + abs(self.fdm["velocities/q-rad_sec"])
        jerk = float(np.sum(np.abs(action - self.prev_action)))

        reward = (
            1.0
            - 0.10 * alt_err_ft
            - 0.50 * tilt
            - 0.10 * spin
            - 0.05 * jerk
        )

        crashed = (
            self.fdm["position/h-agl-ft"] < 1.0
            or self.fdm["position/h-agl-ft"] > self.target_altitude + 60.0
            or abs(self.fdm["attitude/phi-rad"]) > 1.0
            or abs(self.fdm["attitude/theta-rad"]) > 1.0
        )
        if crashed:
            reward -= 50.0

        self.prev_action = action.copy()

        terminated = bool(crashed)
        truncated = bool(self.step_count >= self.max_steps)

        return obs, float(reward), terminated, truncated, {}

Overwriting /content/repo/src/drone_rl/envs/f450_env.py


In [53]:
%%writefile /content/repo/src/drone_rl/train.py
"""F450 hover gorevi icin PPO egitimi + EvalCallback."""

import argparse
from pathlib import Path

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback

from drone_rl.envs.f450_env import F450HoverEnv

def make_env():
    return Monitor(F450HoverEnv())

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--timesteps", type=int, default=300_000)
    ap.add_argument("--n-envs", type=int, default=4)
    ap.add_argument("--out", type=str, default="/content/runs/hover")
    ap.add_argument("--eval-freq", type=int, default=10000)
    args = ap.parse_args()

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)

    # Eğitim Ortamı
    venv = DummyVecEnv([make_env for _ in range(args.n_envs)])
    venv = VecNormalize(venv, norm_obs=True, norm_reward=True, clip_obs=10.0)

    # Değerlendirme Ortamı (Callback için)
    eval_env = DummyVecEnv([make_env])
    # Eğitim ortamının normalizasyon istatistiklerini kullanması için wrap ediyoruz
    eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False, clip_obs=10.0, training=False)

    model = PPO(
        "MlpPolicy", venv,
        n_steps=1024, batch_size=256, n_epochs=10,
        gamma=0.99, gae_lambda=0.95, clip_range=0.2,
        learning_rate=3e-4, ent_coef=0.0,
        verbose=1, device="cpu",
        tensorboard_log=str(out / "tb"),
    )

    # 1. Checkpoint Callback: Periyodik kayıt
    ckpt_cb = CheckpointCallback(
        save_freq=max(20_000 // args.n_envs, 1),
        save_path=str(out / "ckpt"),
        name_prefix="ppo",
    )

    # 2. Eval Callback: En iyi modeli bulma ve test
    eval_cb = EvalCallback(
        eval_env,
        best_model_save_path=str(out / "best_model"),
        log_path=str(out / "logs"),
        eval_freq=max(args.eval_freq // args.n_envs, 1),
        deterministic=True,
        render=False
    )

    model.learn(total_timesteps=args.timesteps, callback=[ckpt_cb, eval_cb])

    model.save(out / "model_final")
    venv.save(str(out / "vecnormalize.pkl"))
    print("Egitim tamamlandi ve kaydedildi:", out)

if __name__ == "__main__":
    main()

Overwriting /content/repo/src/drone_rl/train.py


In [54]:
# EvalCallback ile güncellenmiş yeni eğitimi başlat
!cd /content/repo/src && python -m drone_rl.train \
    --timesteps 300000 \
    --n-envs 4 \
    --eval-freq 10000 \
    --out /content/runs/hover_v2

2026-08-31 12:42:02.003947: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-31 12:42:02.278070: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


     JSBSim Flight Dynamics Model v1.2.4 Feb  7 2026 11:12:49
            [JSBSim-ML v2.0]

JSBSim startup beginning ...


YOU HAVE AN INCOMPATIBLE CFG FILE FOR THIS AIRCRAFT.

In [15]:
#training sürecini başlatır 300.000 step
!cd /content/repo/src && python -m drone_rl.train --timesteps 300000 --n-envs 4 --out /content/runs/hover_v1

2026-08-31 08:00:47.598010: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-31 08:00:47.692590: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


     JSBSim Flight Dynamics Model v1.2.4 Feb  7 2026 11:12:49
            [JSBSim-ML v2.0]

JSBSim startup beginning ...


YOU HAVE AN INCOMPATIBLE CFG FILE FOR THIS AIRCRAFT.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/runs

In [50]:
%%writefile /content/repo/src/drone_rl/evaluate.py
"""Egitilmis politikayi calistir ve ACME telemetry formatinda kaydet."""

import argparse
from pathlib import Path
import numpy as np
import pandas as pd
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from drone_rl.envs.f450_env import F450HoverEnv

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--run", type=str, default="/content/repo/runs/hover_v1")
    ap.add_argument("--episodes", type=int, default=3)
    ap.add_argument("--output", type=str, default="/content/acme_telemetry.csv")
    args = ap.parse_args()

    run = Path(args.run)
    venv = DummyVecEnv([lambda: F450HoverEnv()])
    venv = VecNormalize.load(str(run / "vecnormalize.pkl"), venv)
    venv.training = False
    venv.norm_reward = False

    model = PPO.load(str(run / "model"), device="cpu")
    raw = venv.envs[0]
    all_telemetry = []

    for ep in range(args.episodes):
        obs = venv.reset()
        t = 0.0
        dt = 1.0 / 20.0  # control_hz

        while True:
            action, _ = model.predict(obs, deterministic=True)
            obs, _, done, _ = venv.step(action)

            # ACME Telemetry Data Map with Headers
            data = {
                "timestamp": round(t, 4),
                "episode": ep,
                "pos_x_ft": round(float(raw.fdm["position/h-agl-ft"]), 4),
                "pos_y_ft": 0.0,
                "pos_z_ft": round(float(raw.fdm["position/h-agl-ft"]), 4),
                "roll_rad": round(float(raw.fdm["attitude/phi-rad"]), 6),
                "pitch_rad": round(float(raw.fdm["attitude/theta-rad"]), 6),
                "yaw_rad": round(float(raw.fdm["attitude/psi-true-rad"]), 6),
                "alt_err": round(abs(raw.fdm["position/h-agl-ft"] - raw.target_altitude), 4)
            }
            all_telemetry.append(data)
            t += dt
            if done[0]:
                break #episode length

    df = pd.DataFrame(all_telemetry)
    # Save with headers
    df.to_csv(args.output, index=False, header=True)
    print(f"ACME Telemetry kaydedildi (Headerlar eklendi): {args.output}")
    print(df.head())

if __name__ == '__main__':
    main()

Overwriting /content/repo/src/drone_rl/evaluate.py


In [36]:
#gitignore oluşturur ve günceller
with open("/content/repo/.gitignore", "w") as f:
    f.write("__pycache__/\n.ipynb_checkpoints/\nruns/*/tb/\nruns/*/ckpt/\n")
!cat /content/repo/.gitignore

__pycache__/
.ipynb_checkpoints/
runs/*/tb/
runs/*/ckpt/


In [51]:
# Egitilen modeli test etmek ve sonuclari kaydetmek icin
!cd /content/repo/src && python -m drone_rl.evaluate --run /content/repo/runs/hover_v1 --output /content/iz.csv

2026-08-31 11:27:07.015621: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-31 11:27:07.087577: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


     JSBSim Flight Dynamics Model v1.2.4 Feb  7 2026 11:12:49
            [JSBSim-ML v2.0]

JSBSim startup beginning ...


YOU HAVE AN INCOMPATIBLE CFG FILE FOR THIS AIRCRAFT.

In [52]:
import pandas as pd

# Oluşturulan CSV telemetri dosyasını oku
df_telemetry = pd.read_csv('/content/iz.csv')

# İlk 10 satırı ve genel istatistikleri göster
print("Telemetri Veri Özeti:")
display(df_telemetry.head(10))
print("\nİstatistiksel Özet:")
display(df_telemetry.describe())

Telemetri Veri Özeti:


,timestamp,episode,pos_x_ft,pos_y_ft,pos_z_ft,roll_rad,pitch_rad,yaw_rad,alt_err
0,0.00,0,31.2578,0.0,31.2578,0.048057,0.015536,6.283185,1.2578
1,0.05,0,31.1410,0.0,31.1410,0.048057,0.015536,6.283185,1.1410
2,0.10,0,30.9575,0.0,30.9575,0.048057,0.015536,6.283185,0.9575
3,0.15,0,30.7447,0.0,30.7447,0.048057,0.015536,6.283185,0.7447
4,0.20,0,30.5623,0.0,30.5623,0.048057,0.015536,6.283185,0.5623
5,0.25,0,30.4508,0.0,30.4508,0.048057,0.015536,6.283185,0.4508
6,0.30,0,30.4029,0.0,30.4029,0.048057,0.015536,6.283185,0.4029
7,0.35,0,30.3825,0.0,30.3825,0.048057,0.015536,6.283185,0.3825
8,0.40,0,30.3589,0.0,30.3589,0.048057,0.015536,6.283185,0.3589
9,0.45,0,30.3193,0.0,30.3193,0.048057,0.015536,6.283185,0.3193



İstatistiksel Özet:


,timestamp,episode,pos_x_ft,pos_y_ft,pos_z_ft,roll_rad,pitch_rad,yaw_rad,alt_err
count,1200.000000,1200.000000,1200.00000,1200.0,1200.00000,1200.000000,1200.000000,1200.000000,1200.000000
mean,9.975000,1.000000,29.99174,0.0,29.99174,0.037813,-0.013230,2.167699,0.047992
std,5.775892,0.816837,0.12298,0.0,0.12298,0.009398,0.023590,2.988072,0.113522
min,0.000000,0.000000,28.64370,0.0,28.64370,-0.042087,-0.042112,0.000000,0.000000
25%,4.987500,0.000000,29.96890,0.0,29.96890,0.026104,-0.042105,0.000000,0.011375
50%,9.975000,1.000000,30.00055,0.0,30.00055,0.039624,-0.013297,0.000000,0.028200
75%,14.962500,2.000000,30.01540,0.0,30.01540,0.048049,0.015534,6.283185,0.046100
max,19.950000,2.000000,31.25780,0.0,31.25780,0.048057,0.048149,6.283185,1.356300


In [16]:
%%writefile /content/repo/configs/ppo_hover.yaml
env:
  target_altitude_ft: 30.0
  episode_seconds: 20.0
  control_hz: 20
  physics_hz: 240
  hover_throttle: 0.410
  throttle_range: 0.25
ppo:
  policy: MlpPolicy
  n_steps: 1024
  batch_size: 256
  n_epochs: 10
  gamma: 0.99
  gae_lambda: 0.95
  clip_range: 0.2
  learning_rate: 0.0003
  ent_coef: 0.0
train:
  timesteps: 300000
  n_envs: 4

Overwriting /content/repo/configs/ppo_hover.yaml


In [39]:
readme = """# quadcopter-rl-copilot

JSBSim F450 quadcopter modeli uzerinde PPO ile hover kontrolu.

## Sonuc (hover_v1)

300.000 adim egitim sonrasi, 5 degerlendirme episode'unda:

- Episode uzunlugu: 400/400 (hic dusme yok)
- Hedef irtifadan ortalama sapma: 0.02 - 0.06 ft
- Ortalama egilme: 0.02 - 0.08 rad

## Kurulum (Colab)

    !pip install -q stable-baselines3 gymnasium
    !pip uninstall -y -q jsbsim
    !pip install -q jsbsim==1.2.4

Repoyu klonladiktan sonra src dizinini Python yoluna ekle:

    import os, sys
    sys.path.insert(0, "/content/repo/src")
    os.environ["PYTHONPATH"] = "/content/repo/src"

## Kullanim

Egitim:

    cd src && python -m drone_rl.train --timesteps 300000 --n-envs 4 --out ../runs/hover_v2

Degerlendirme:

    cd src && python -m drone_rl.evaluate --run ../runs/hover_v1 --csv /content/iz.csv

## Yapi

- src/drone_rl/envs/f450_env.py - Gymnasium ortami
- src/drone_rl/train.py - PPO egitimi
- src/drone_rl/evaluate.py - egitilmis politikanin olculmesi
- configs/ppo_hover.yaml - kullanilan ayarlar
- runs/hover_v1/ - egitilmis model ve normalizasyon istatistikleri
- notebooks/quadcopter_rl.ipynb - Colab calisma defteri

## Notlar

- JSBSim emperyal birim kullanir (ft, lbs, fps).
- Hover gazi 0.410 olarak olculdu. Aksiyon bu deger etrafinda +-0.25
  araliginda olceklenir, boylece sifir aksiyon "asili kal" anlamina gelir.
- vecnormalize.pkl model ile birlikte yuklenmelidir, aksi halde politika
  yanlis olcekli gozlem alir ve calismaz.
- F450 XML'i yuklenirken "version 3.0" uyarisi verir; zararsizdir.
"""

with open("/content/repo/README.md", "w") as f:
    f.write(readme)

print(open("/content/repo/README.md").read()[:300])

# quadcopter-rl-copilot

JSBSim F450 quadcopter modeli uzerinde PPO ile hover kontrolu.

## Sonuc (hover_v1)

300.000 adim egitim sonrasi, 5 degerlendirme episode'unda:

- Episode uzunlugu: 400/400 (hic dusme yok)
- Hedef irtifadan ortalama sapma: 0.02 - 0.06 ft
- Ortalama egilme: 0.02 - 0.08 rad

#


In [17]:
%%bash
# Repo dizinine geç
cd /content/repo

# Tüm değişiklikleri ekle
git add -A

# Değişiklikleri açıkla (Eğer içeride değişiklik varsa)
if ! git diff-index --quiet HEAD --; then
  git commit -m "ACMI"
  git push origin main
  echo "Değişiklikler başarıyla pushlandı."
else
  echo "Pushlanacak bir değişiklik bulunamadı."
fi

[main f4a7acc] Güncel telemetry ve değerlendirme değişiklikleri
 3 files changed, 36 insertions(+), 37 deletions(-)
Değişiklikler başarıyla pushlandı.


To https://github.com/miray7yuce/quadcopter-rl-copilot.git
   fbb7bfa..f4a7acc  main -> main


In [55]:
%%bash
cd /content/repo
git add -A
if ! git diff-index --quiet HEAD --; then
  git commit -m "Added EvalCallback to training pipeline and updated configs"
  git push origin main
  echo "Tüm değişiklikler başarıyla GitHub'a gönderildi."
else
  echo "Pushlanacak yeni bir değişiklik bulunamadı."
fi

[main be8e683] Added EvalCallback to training pipeline and updated configs
 3 files changed, 100 insertions(+), 23 deletions(-)
Tüm değişiklikler başarıyla GitHub'a gönderildi.


To https://github.com/miray7yuce/quadcopter-rl-copilot.git
   f4a7acc..be8e683  main -> main
